# Cleaning3
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd

client = DatalakeClient()

# Get the files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
file_codes = ['ADSP_PHC_BIOMARKER']
#'ADNIMERGE', 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 'BLCHANGE', 'DXSUM', 
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL'

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_02', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

## Operazioni
- Trasformare i volumi come percentuali di ICV

In [4]:
# create new support file with the info from the new dataset after cleaning 2
support_file_path = 'ADNI_variables_cleaned2'
support_file = pd.read_excel(support_file_path+'.xlsx')
new_name = 'ADNI_variables_cleaned3'

In [5]:
if os.path.isfile(new_name+'.xlsx'):
    update_new_support_file(support_file, new_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_name, rename_column=False)

The ADNI_variables_cleaned3 file has been updated with the new file_code: ['UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', nan, 'ADNI_DIAN_COMPARISON']
Open the file and verify it, if needed update the variables names and metadata
The ADNI_variables_cleaned3 file has restored the previous information of the file_code: ['ADSP_PHC_BIOMARKER']
Open the file and verify it, if needed update the variables names and metadata


In [6]:
dataCleaner = DataCleaner(support_file_path=new_name+'.xlsx')
new_support_file = pd.read_excel(new_name+'.xlsx')

In [7]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    df_new = df.copy(deep=True)
    #new_support_file = dataCleaner.update_self_support_file(new_support_file)
    file_code, metadat_custom = dataCleaner.get_file_code_metadata(file_name, prefix='cleaned/single_file')
    if 'volume' in support_file[support_file['file_code'] == file_code]['metadati_normalizzazione'].values:
        print('\n\n ----', file_name)
        # verifica che i volumi siano già totali e non solo una parte laterale
        processed_df, volume_list = dataCleaner.get_volumes_total(df_new, file_code)
        # Transform volumes as ICV percentage
        final_df = dataCleaner.transform_volumes_as_ICV_percent(processed_df, volume_list, file_code)
    else:
        final_df = df_new
    # aggiunta di righe per i nuovi parametri e rimozione dal support di variabili non più nel df
    new_support_file = update_variables_support_file(final_df, new_support_file, file_code)

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_03', file_name=file_name, prefix='cleaned/single_file', updated_support_file=new_support_file)
    # get info into the new support file
    infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name, prefix='cleaned/single_file')
    for key in final_df.keys():
        if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
            new_support_file = infoSupportFile.get_varible_info(key)

    # Create new file name for datalake
    new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_03')


    #upload the new file
    result = client.upload_dataframe(
        df=final_df,
        object_name=new_file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )
   
    
save_df(df_to_save=new_support_file, output_path=new_name)     

In [8]:
final_df

,RID,COHORT,VISIT_MONTH,EXAMDATE,AGE,EDUCAT,AB42,AB42_z,TTAU,TTAU_z,...,GENDER/male,ETHNICITY/latino,ETHNICITY/not_latino,RACE/Asian,RACE/Black,RACE/Native_american,RACE/White,DX/CN,DX/Dementia,DX/MCI
0,3,ADNI1,0,2006-09-13,82.37,18,137.0,-0.589048,76.5,-0.077397,...,True,False,True,False,False,False,True,False,True,False
1,4,ADNI1,0,2006-11-28,68.91,10,246.0,1.284729,49.4,-0.898866,...,True,True,False,False,False,False,True,False,False,True
2,5,ADNI1,0,2006-09-06,74.77,16,136.0,-0.612499,120.0,0.768228,...,True,False,True,False,False,False,True,True,False,False
3,8,ADNI1,0,2006-09-20,85.56,18,228.0,1.041489,94.6,0.321497,...,False,False,True,False,False,False,True,True,False,False
4,10,ADNI1,0,2006-11-10,74.94,12,92.8,-1.835993,82.6,0.066706,...,False,False,True,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1240,5289,ADNI2,0,2013-11-18,60.05,16,232.0,1.097163,41.9,-1.208160,...,False,False,True,False,False,False,True,True,False,False
1241,5290,ADNI2,0,2013-09-27,67.16,12,128.0,-0.806566,148.0,1.162152,...,False,False,True,False,False,False,True,True,False,False
1242,5292,ADNI2,0,2013-11-13,74.54,13,154.0,-0.214607,110.0,0.604792,...,False,False,True,False,False,False,True,True,False,False
1243,5295,ADNI2,0,2013-12-17,75.63,15,117.0,-1.094207,20.5,-2.550904,...,False,False,True,False,False,False,True,True,False,False


In [9]:
updated_metadata

{'cofattori': ['EDUCAT',
  'GENDER/female',
  'GENDER/male',
  'ETHNICITY/latino',
  'ETHNICITY/not_latino',
  'RACE/Asian',
  'RACE/Black',
  'RACE/Native_american',
  'RACE/White'],
 'file_code': 'ADSP_PHC_BIOMARKER',
 'level': 'cleaned_03',
 'norm_intervallo': ['AB42', 'AB42_z', 'TTAU', 'TTAU_z', 'PTAU', 'PTAU_z'],
 'norm_scala': ['Tprofile', 'Aprofile'],
 'norm_scale_value': {},
 'norm_volume': [],
 'population': ['ADNI1', 'ADNIGO', 'ADNI2'],
 'predittori': ['AB42',
  'AB42_z',
  'TTAU',
  'TTAU_z',
  'PTAU',
  'PTAU_z',
  'Tprofile',
  'Aprofile'],
 'source': 'ADNI',
 'volume_norm_values': []}